In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

DEVICE= torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE")


DEVICE


In [3]:
# 1. 전처리: Pre-trained 모델을 사용할 때는 ImageNet 정규화 값을 쓰는 것이 정석입니다.
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

In [4]:
# 2. 데이터 로드
train_DS = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_DS = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_DL = DataLoader(train_DS, batch_size=128, shuffle=True)
test_DL = DataLoader(test_DS, batch_size=128, shuffle=False)


c:\Users\darwi\.conda\envs\env1\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [5]:
# 3. 모델 로드 및 수정
# weights='DEFAULT'는 가장 최신의 ImageNet 가중치를 불러옵니다.
model = torchvision.models.vgg16(weights='DEFAULT')
model.avgpool = nn.AdaptiveAvgPool2d((1, 1))

# CIFAR-10(32x32) 이미지는 VGG16의 Feature Extractor를 통과하면 1x1x512가 됩니다.
# 원본 VGG16은 7x7x512를 기대하므로, 분류기의 첫 레이어 입력 크기를 수정합니다.
model.classifier[0] = nn.Linear(512 * 1 * 1, 4096)
model.classifier[6] = nn.Linear(4096, 10) # 최종 출력 10개

model = model.to(DEVICE)

In [6]:
# 4. 손실 함수 및 옵티마이저
criterion = nn.CrossEntropyLoss()
# Fine-tuning 시에는 이미 학습된 가중치를 망가뜨리지 않기 위해 
# SGD를 사용하고 학습률(lr)을 아주 작게(1e-3 이하) 가져가는 것이 핵심입니다.
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

In [7]:
# 5. Fine-tuning 학습 루프
def train_finetune(model, loader, epochs):
    model.train()
    for ep in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        scheduler.step()
        acc = 100. * correct / total
        print(f"Epoch [{ep+1}/{epochs}] Loss: {running_loss/len(loader):.4f} | Acc: {acc:.2f}%")

# 6. 실행
train_finetune(model, train_DL, epochs=20)

Epoch [1/20] Loss: 1.7676 | Acc: 33.71%
Epoch [2/20] Loss: 0.8048 | Acc: 72.43%
Epoch [3/20] Loss: 0.6339 | Acc: 78.88%
Epoch [4/20] Loss: 0.5502 | Acc: 81.55%
Epoch [5/20] Loss: 0.5004 | Acc: 83.26%
Epoch [6/20] Loss: 0.4602 | Acc: 84.53%
Epoch [7/20] Loss: 0.4279 | Acc: 85.76%
Epoch [8/20] Loss: 0.4038 | Acc: 86.48%
Epoch [9/20] Loss: 0.3791 | Acc: 87.30%
Epoch [10/20] Loss: 0.3668 | Acc: 87.70%
Epoch [11/20] Loss: 0.3455 | Acc: 88.33%
Epoch [12/20] Loss: 0.3267 | Acc: 88.87%
Epoch [13/20] Loss: 0.3195 | Acc: 89.26%
Epoch [14/20] Loss: 0.3079 | Acc: 89.63%
Epoch [15/20] Loss: 0.3020 | Acc: 89.84%
Epoch [16/20] Loss: 0.2937 | Acc: 89.84%
Epoch [17/20] Loss: 0.2863 | Acc: 90.13%
Epoch [18/20] Loss: 0.2824 | Acc: 90.30%
Epoch [19/20] Loss: 0.2799 | Acc: 90.39%
Epoch [20/20] Loss: 0.2777 | Acc: 90.54%


In [8]:
def evaluate_model(model, test_loader, criterion, device):
    model.eval() # 1. 평가 모드 전환 (Dropout, BN 비활성화)
    test_loss = 0
    correct = 0
    total = 0
    
    # 2. 기울기 계산 비활성화 (메모리 절약 및 속도 향상)
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            
            # 3. 모델 추론 (Forward)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            test_loss += loss.item() * images.size(0)
            
            # 4. 정확도 계산
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    # 최종 지표 계산
    avg_loss = test_loss / total
    accuracy = 100 * correct / total
    
    print("-" * 30)
    print(f"Test Loss: {avg_loss:.4f}")
    print(f"Test Accuracy: {accuracy:.2f} %")
    print("-" * 30)
    
    return avg_loss, accuracy

# 실행 코드
test_loss, test_acc = evaluate_model(model, test_DL, criterion, DEVICE)

------------------------------
Test Loss: 0.3543
Test Accuracy: 88.62 %
------------------------------
